# Neural Chunking: BiLSTM vs Transformer

Sequence labelling for shallow parsing in PyTorch. A bidirectional LSTM is compared against a Transformer encoder built from scratch (no `nn.Transformer`, no `transformers` library). Each model predicts one of 17 CoNLL-style chunk tags per token.

## Setup

In [ ]:
import os
import math
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)

SEED = 72
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load and explore data

In [ ]:
DATA_PATH = 'data/chunk_dataset.txt'

def load_chunks(path):
    """Parse a CoNLL-style file into a list of (words, tags) per sentence."""
    with open(path, 'r', encoding='utf-8') as f:
        raw = f.read()
    sentences = []
    for block in raw.strip().split('\n\n'):
        words, tags = [], []
        for line in block.split('\n'):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                words.append(parts[0])
                tags.append(parts[-1])
        if words:
            sentences.append((words, tags))
    return sentences

sentences = load_chunks(DATA_PATH)
lengths = [len(s[0]) for s in sentences]
tag_counter = Counter(t for _, tags in sentences for t in tags)

print(f'Total sentences: {len(sentences)}')
print(f'Sentence length: min={min(lengths)} median={int(np.median(lengths))} '
      f'mean={np.mean(lengths):.1f} max={max(lengths)}')
print(f'Unique tags: {len(tag_counter)}')
for t, c in tag_counter.most_common():
    print(f'  {t:<8s} {c}')

## Train / validation / test split

In [ ]:
train_val, test = train_test_split(sentences, test_size=0.10, random_state=SEED)
train, val = train_test_split(train_val, test_size=1/9, random_state=SEED)

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

## Build vocabularies

In [ ]:
PAD_TOKEN, UNK_TOKEN = '<PAD>', '<UNK>'
PAD_IDX, UNK_IDX = 0, 1

def build_word_vocab(train_sentences, min_freq=1):
    """Build a word-to-index map from training sentences with PAD and UNK reserved."""
    counter = Counter()
    for words, _ in train_sentences:
        counter.update(words)
    word2idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for w, c in counter.items():
        if c >= min_freq:
            word2idx[w] = len(word2idx)
    return word2idx

TAG_LIST = ['B-NP','I-NP','B-ADVP','I-ADVP','B-CONJP','I-CONJP',
            'B-ADJP','I-ADJP','B-VP','I-VP','B-PP','I-PP',
            'B-SBAR','I-SBAR','B-INTJ','B-PRT','O']
TAG_PAD_IDX = 0
tag2idx = {'<TAG_PAD>': TAG_PAD_IDX}
for t in TAG_LIST:
    tag2idx[t] = len(tag2idx)
idx2tag = {i: t for t, i in tag2idx.items()}

word2idx = build_word_vocab(train)
print(f'Word vocab size: {len(word2idx)}')
print(f'Tag vocab size:  {len(tag2idx)}')

## Dataset and DataLoader

In [ ]:
class ChunkDataset(Dataset):
    """Wrap (words, tags) sentences and yield (word_ids, tag_ids) tensors."""
    def __init__(self, sentences, word2idx, tag2idx):
        self.sentences = sentences
        self.word2idx = word2idx
        self.tag2idx = tag2idx

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, i):
        words, tags = self.sentences[i]
        x = [self.word2idx.get(w, UNK_IDX) for w in words]
        y = [self.tag2idx[t] for t in tags]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def collate(batch):
    """Pad a batch to its longest sentence using PAD_IDX for inputs and TAG_PAD_IDX for tags."""
    lengths = torch.tensor([len(x) for x, _ in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    xs = torch.full((len(batch), max_len), PAD_IDX, dtype=torch.long)
    ys = torch.full((len(batch), max_len), TAG_PAD_IDX, dtype=torch.long)
    for i, (x, y) in enumerate(batch):
        xs[i, :len(x)] = x
        ys[i, :len(y)] = y
    return xs, ys, lengths


BATCH_SIZE = 64
train_loader = DataLoader(ChunkDataset(train, word2idx, tag2idx),
                          batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(ChunkDataset(val,   word2idx, tag2idx),
                          batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(ChunkDataset(test,  word2idx, tag2idx),
                          batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

xs, ys, lens = next(iter(train_loader))
print(f'Batch shapes -> xs: {tuple(xs.shape)} ys: {tuple(ys.shape)} lens: {tuple(lens.shape)}')

## BiLSTM model

In [ ]:
class BiLSTMTagger(nn.Module):
    """Embedding to bidirectional LSTM to per-token linear classifier over tags."""
    def __init__(self, vocab_size, num_tags,
                 emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_tags)

    def forward(self, x, lengths=None):
        emb = self.dropout(self.embedding(x))
        out, _ = self.lstm(emb)
        return self.fc(self.dropout(out))

## Transformer model (from scratch)

In [ ]:
class PositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding added to token embeddings."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MultiHeadSelfAttention(nn.Module):
    """Scaled dot-product self-attention over multiple heads with optional padding mask."""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, D = x.shape
        Q = self.q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.k(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.v(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        ctx = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T, D)
        return self.out(ctx)


class TransformerEncoderLayer(nn.Module):
    """One encoder block: masked self-attention then feed-forward, each with residual and LayerNorm."""
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.attn(x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


class TransformerTagger(nn.Module):
    """Embedding plus positional encoding, stacked encoder layers, then per-token linear head."""
    def __init__(self, vocab_size, num_tags,
                 d_model=128, num_heads=4, num_layers=3,
                 ff_dim=256, dropout=0.1, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model, max_len=max_len)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_tags)

    def forward(self, x, lengths=None):
        pad_mask = (x != PAD_IDX).unsqueeze(1).unsqueeze(2)
        h = self.dropout(self.pos(self.embedding(x)))
        for layer in self.layers:
            h = layer(h, mask=pad_mask)
        return self.fc(h)

## Training loop and plotting

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Run one pass over loader; train if an optimizer is given, otherwise evaluate."""
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    grad_ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with grad_ctx:
        for xs, ys, lengths in loader:
            xs, ys = xs.to(device), ys.to(device)
            logits = model(xs, lengths)
            loss = criterion(logits.reshape(-1, logits.size(-1)),
                             ys.reshape(-1))
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            mask = ys != TAG_PAD_IDX
            preds = logits.argmax(-1)
            total_correct += ((preds == ys) & mask).sum().item()
            total_tokens  += mask.sum().item()
            total_loss    += loss.item() * xs.size(0)
    return total_loss / len(loader.dataset), total_correct / max(total_tokens, 1)


def train_model(model, name, epochs=10, lr=1e-3):
    """Train with Adam and CrossEntropyLoss, print per-epoch metrics, and return the history dict."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=TAG_PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  []}
    for epoch in range(1, epochs + 1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader,   criterion)
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        print(f'[{name}] Epoch {epoch:02d} | '
              f'train_loss {tl:.4f} acc {ta:.4f} | '
              f'val_loss {vl:.4f} acc {va:.4f}')
    return history


def plot_history(history, name):
    """Plot loss and token accuracy curves for train and val side by side."""
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    epochs_axis = range(1, len(history['train_loss']) + 1)
    ax[0].plot(epochs_axis, history['train_loss'], marker='o', label='train')
    ax[0].plot(epochs_axis, history['val_loss'],   marker='o', label='val')
    ax[0].set_title(f'{name}: Loss vs epoch')
    ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(epochs_axis, history['train_acc'], marker='o', label='train')
    ax[1].plot(epochs_axis, history['val_acc'],   marker='o', label='val')
    ax[1].set_title(f'{name}: Token accuracy vs epoch')
    ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy'); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## Train BiLSTM

In [ ]:
EPOCHS = 10

torch.manual_seed(SEED)
lstm_model = BiLSTMTagger(len(word2idx), len(tag2idx))
n_params_lstm = sum(p.numel() for p in lstm_model.parameters())
print(f'BiLSTM parameters: {n_params_lstm:,}')
lstm_history = train_model(lstm_model, 'BiLSTM', epochs=EPOCHS, lr=1e-3)
plot_history(lstm_history, 'BiLSTM')

## Train Transformer

In [ ]:
torch.manual_seed(SEED)
trans_model = TransformerTagger(len(word2idx), len(tag2idx),
                                 d_model=128, num_heads=4,
                                 num_layers=3, ff_dim=256, dropout=0.1)
n_params_trans = sum(p.numel() for p in trans_model.parameters())
print(f'Transformer parameters: {n_params_trans:,}')
trans_history = train_model(trans_model, 'Transformer', epochs=EPOCHS, lr=1e-3)
plot_history(trans_history, 'Transformer')

## Compare models on validation

In [ ]:
def collect_preds(model, loader):
    """Return (true, pred) arrays of token-level labels with pad positions removed."""
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for xs, ys, lengths in loader:
            xs, ys = xs.to(device), ys.to(device)
            preds = model(xs).argmax(-1)
            mask = ys != TAG_PAD_IDX
            all_preds.append(preds[mask].cpu().numpy())
            all_true.append(ys[mask].cpu().numpy())
    return np.concatenate(all_true), np.concatenate(all_preds)


tag_label_indices = list(range(1, len(tag2idx)))
tag_label_names   = [idx2tag[i] for i in tag_label_indices]

val_results = []
for name, model in [('BiLSTM', lstm_model), ('Transformer', trans_model)]:
    yt, yp = collect_preds(model, val_loader)
    val_results.append({
        'model':       name,
        'accuracy':    accuracy_score(yt, yp),
        'macro_f1':    f1_score(yt, yp, labels=tag_label_indices, average='macro',    zero_division=0),
        'weighted_f1': f1_score(yt, yp, labels=tag_label_indices, average='weighted', zero_division=0),
    })

print(f"{'Model':<14}{'Accuracy':>12}{'Macro F1':>12}{'Weighted F1':>14}")
for r in val_results:
    print(f"{r['model']:<14}{r['accuracy']:>12.4f}{r['macro_f1']:>12.4f}{r['weighted_f1']:>14.4f}")

best_row = max(val_results, key=lambda r: r['macro_f1'])
best_name  = best_row['model']
best_model = lstm_model if best_name == 'BiLSTM' else trans_model
print(f'\nBest model on validation (by macro F1): {best_name}')

## Evaluate best model on test set

In [ ]:
y_true, y_pred = collect_preds(best_model, test_loader)

test_acc         = accuracy_score(y_true, y_pred)
test_macro_f1    = f1_score(y_true, y_pred, labels=tag_label_indices, average='macro',    zero_division=0)
test_weighted_f1 = f1_score(y_true, y_pred, labels=tag_label_indices, average='weighted', zero_division=0)

print(f'=== Test results: {best_name} ===')
print(f'Accuracy:    {test_acc:.4f}')
print(f'Macro F1:    {test_macro_f1:.4f}')
print(f'Weighted F1: {test_weighted_f1:.4f}')
print()
print(classification_report(y_true, y_pred,
                            labels=tag_label_indices,
                            target_names=tag_label_names,
                            zero_division=0,
                            digits=4))

cm = confusion_matrix(y_true, y_pred, labels=tag_label_indices)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=tag_label_names, yticklabels=tag_label_names,
            ax=ax, cbar=True)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'{best_name}: Confusion matrix on test set')
plt.tight_layout(); plt.show()